# 04. Data Cleaning, Missing Values & Type Hygiene: Beginner Guide

### 📌 Overview
Master **04. Data Cleaning, Missing Values & Type Hygiene: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Null Detection & Handling**: Covers `.isnull()`/`.isna()`, `.notnull()`/`.notna()`, `.dropna()`, and `.fillna()`.
- **Deduplication**: Covers `.duplicated()` and `.drop_duplicates()`.
- **Type Hygiene & Conversion**: Covers `.astype()`, `pd.to_numeric()`, and `pd.to_datetime()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount card_type  \
0       TX109326      C55082       M3549              607.78      Visa   
1       TX106376      C76616       M3068             1819.11      Visa   

  transaction_status device_type  account_age_months     transaction_date  \
0           Reversed      Mobile                   8  2026-02-17 08:28:57   
1            Pending         POS                  28          03-Jan-2025   

  region  is_fraud  
0  North         0  
1   West         1  


### 🔹 Detecting Nulls: `.isnull()` / `.isna()`
- **What it does:** Counts missing entries across all columns in raw_transactions.csv.
- **Syntax:** `df.isnull().sum()`
- **Operation:** `print('Missing Values Count per Column:\n', df.isnull().sum())`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [2]:
print('Missing Values Count per Column:\n', df.isnull().sum())

Missing Values Count per Column:
 transaction_id          0
customer_id             0
merchant_id             0
transaction_amount    749
card_type               0
transaction_status      0
device_type             0
account_age_months      0
transaction_date        0
region                  0
is_fraud                0
dtype: int64


### 🔹 Detecting Non-Nulls: `.notnull()` / `.notna()`
- **What it does:** Filters rows with complete customer IDs.
- **Syntax:** `df[df['customer_id'].notnull()]`
- **Operation:** `print('Valid Customer Rows Count:', df['customer_id'].notnull().sum())`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [3]:
print('Valid Customer Rows Count:', df['customer_id'].notnull().sum())

Valid Customer Rows Count: 15000


### 🔹 Dropping Missing Values: `.dropna()`
- **What it does:** Removes rows missing transaction amounts or dates.
- **Syntax:** `df.dropna(subset=['transaction_amount', 'transaction_date'])`
- **Operation:** `cleaned_df = df.dropna(subset=['transaction_amount'])`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [4]:
cleaned_df = df.dropna(subset=['transaction_amount'])
print('Rows after dropna on amount:', len(cleaned_df))

Rows after dropna on amount: 14251


### 🔹 Imputing Missing Values: `.fillna()`
- **What it does:** Imputes missing numeric transaction amounts with the median amount.
- **Syntax:** `df['transaction_amount'].fillna(df['transaction_amount'].median())`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [5]:
imputed_amt = df['transaction_amount'].fillna(df['transaction_amount'].median())
print('Null count after median imputation:', imputed_amt.isnull().sum())

Null count after median imputation: 0


### 🔹 Detecting Duplicates: `.duplicated()`
- **What it does:** Identifies duplicate transaction records.
- **Syntax:** `df.duplicated(subset=['transaction_id'])`
- **Operation:** `duplicate_count = df.duplicated(subset=['transaction_id']).sum()`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [6]:
duplicate_count = df.duplicated(subset=['transaction_id']).sum()
print('Duplicate Transaction IDs Count:', duplicate_count)

Duplicate Transaction IDs Count: 100


### 🔹 Removing Duplicates: `.drop_duplicates()`
- **What it does:** Deduplicates raw transactions by transaction_id.
- **Syntax:** `df.drop_duplicates(subset=['transaction_id'], keep='first')`
- **Operation:** `deduped_df = df.drop_duplicates(subset=['transaction_id'], keep='first')`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [7]:
deduped_df = df.drop_duplicates(subset=['transaction_id'], keep='first')
print(f'Original Rows: {len(df)} -> Deduplicated Rows: {len(deduped_df)}')

Original Rows: 15000 -> Deduplicated Rows: 14900


### 🔹 Type Casting with `.astype()`
- **What it does:** Downcasts `is_fraud` from int64 to int8 and `transaction_amount` to float32.
- **Syntax:** `df.astype({'is_fraud': 'int8', 'transaction_amount': 'float32'})`
- **Key Note:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.

In [8]:
optimized_df = deduped_df.astype({'is_fraud': 'int8', 'transaction_amount': 'float32'})
print('Optimized Dtypes:\n', optimized_df.dtypes[['transaction_amount', 'is_fraud']])

Optimized Dtypes:
 transaction_amount    float32
is_fraud                 int8
dtype: object


### 🔹 Robust Numeric Parsing: `pd.to_numeric()`
- **What it does:** Coerces corrupt strings to NaN safely.
- **Syntax:** `pd.to_numeric(df['transaction_amount'], errors='coerce')`
- **Operation:** `coerced_nums = pd.to_numeric(df['transaction_amount'], errors='coerce')`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [9]:
coerced_nums = pd.to_numeric(df['transaction_amount'], errors='coerce')
print('Parsed Numeric Count:', coerced_nums.count())

Parsed Numeric Count: 14251


### 🔹 Robust Datetime Parsing: `pd.to_datetime()`
- **What it does:** Parses mixed string timestamps in raw_transactions.csv into `datetime64[ns]`.
- **Syntax:** `pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')`
- **Key Note:** Ensure your column is converted to datetime with `pd.to_datetime()` before calling `.dt` properties like `.dt.year` or `.dt.day_name()`.

In [10]:
clean_dates = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
print('Parsed Datetime Series Head:\n', clean_dates.head())

Parsed Datetime Series Head:
 0   2026-02-17 08:28:57
1   2025-01-03 00:00:00
2   2025-12-20 00:00:00
3   2026-02-06 03:39:02
4   2025-01-07 00:23:37
Name: transaction_date, dtype: datetime64[ns]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: End-to-End Data Cleaning Pipeline
- **Objective:** Q1: End-to-End Data Cleaning Pipeline
- **Approach:** Build an end-to-end cleaning pipeline on raw_transactions.csv.
- **Syntax:** `df.drop_duplicates().dropna().assign(...)`

In [11]:
pipeline_df = (
    df
    .drop_duplicates(subset=['transaction_id'])
    .assign(
        transaction_date=lambda d: pd.to_datetime(d['transaction_date'], format='mixed', errors='coerce'),
        transaction_amount=lambda d: pd.to_numeric(d['transaction_amount'], errors='coerce')
    )
    .dropna(subset=['transaction_amount', 'transaction_date'])
)
print(f'Cleaned Dataset Ready for Modeling: {len(pipeline_df)} valid transactions')

Cleaned Dataset Ready for Modeling: 14157 valid transactions